# Nest HVAC Analysis: Window Replacement Impact

This notebook tells the story of a window replacement on October 5, 2026, detected through HVAC runtime data from two Nest thermostats — one Upstairs (`09AA01AF171903L1`) and one Downstairs (`15AA01AF062008H1`).

## What you'll find
1. **Data verification** — confirming completeness and coverage
2. **Humidity trends** — indoor and outdoor humidity by device
3. **Cooling efficiency metric** — how we measure HVAC performance
4. **Change point detection** — pinpointing when efficiency shifted
5. **Before vs after analysis** — what the window replacement changed
6. **Key findings** — what actually improved and what didn't

## Data source
The `bronze_nest_hvac_runtime` table contains 15-minute interval readings from two Nest thermostats, covering Aug 1 – Oct 13, 2026. Data was ingested from CSV-quoted JSONL exports via the bronze ingestion notebook.

In [0]:
# Widget parameters — same table as the bronze ingestion notebook
dbutils.widgets.text("catalog", "workspace", "Unity Catalog catalog")
dbutils.widgets.text("schema", "barronfuentes", "Unity Catalog schema")
dbutils.widgets.text("bronze_table", "bronze_nest_hvac_runtime", "Bronze table name")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
bronze_table = dbutils.widgets.get("bronze_table")

full_table_name = f"{catalog}.{schema}.{bronze_table}"
print(f"Source table: {full_table_name}")

## 1. Data Verification

Before any analysis, we verify the table has complete coverage — equal record counts and date ranges per device. Gaps in the data would skew any trend analysis.

**Why this matters:** The Nest data was ingested from monthly JSONL exports. Each file covers one device-month, so a missing file would create a gap that looks like a “change” in any trend chart.

In [0]:
# Verify the bronze table: row count per device and date range
spark.sql(f"""
SELECT device_id,
       COUNT(*) as records,
       MIN(interval_start) as earliest,
       MAX(interval_end) as latest
FROM {full_table_name}
GROUP BY device_id
ORDER BY device_id
""").display()

## 2. Humidity Trends

We start with humidity because it's a direct sensor reading — no derived metric needed. Plotting indoor humidity by device at hourly granularity reveals daily cycles and device differences.

**Aggregation choice:** We aggregate 15-minute readings to hourly averages (~3,500 rows) to stay under the visualization's row limit. The raw data has 14,000+ rows, which the chart truncates.

The second chart compares indoor vs outdoor humidity using a **long-format** query with `UNION ALL` — indoor and outdoor readings as separate rows, labeled by a `series` column that combines device ID and humidity type. This lets the chart color each line independently.

In [0]:
spark.sql(f"""
SELECT device_id,
       date_trunc('hour', CAST(interval_start AS TIMESTAMP)) AS hour_start,
       AVG(indoor_humidity) AS avg_indoor_humidity
FROM {full_table_name}
WHERE indoor_humidity IS NOT NULL
GROUP BY device_id, date_trunc('hour', CAST(interval_start AS TIMESTAMP))
ORDER BY device_id, hour_start
""").display()

Databricks visualization. Run in Databricks to view.

In [0]:
spark.sql(f"""
SELECT CONCAT(device_id, ' — ', humidity_type) AS series,
       hour_start,
       avg_humidity
FROM (
  SELECT device_id,
         date_trunc('hour', CAST(interval_start AS TIMESTAMP)) AS hour_start,
         'indoor' AS humidity_type,
         AVG(indoor_humidity) AS avg_humidity
  FROM {full_table_name}
  WHERE indoor_humidity IS NOT NULL
  GROUP BY device_id, date_trunc('hour', CAST(interval_start AS TIMESTAMP))
  UNION ALL
  SELECT device_id,
         date_trunc('hour', CAST(interval_start AS TIMESTAMP)) AS hour_start,
         'outdoor' AS humidity_type,
         AVG(CAST(outdoor_humidity AS DOUBLE)) AS avg_humidity
  FROM {full_table_name}
  WHERE outdoor_humidity IS NOT NULL
  GROUP BY device_id, date_trunc('hour', CAST(interval_start AS TIMESTAMP))
)
ORDER BY series, hour_start
""").display()

Databricks visualization. Run in Databricks to view.

## 3. Defining a Cooling Efficiency Metric

To compare HVAC performance over time, we need a single metric that captures how hard the system is working relative to the conditions it’s fighting.

**The metric:**
```
cooling_efficiency = avg(outdoor_temp − indoor_temp) / cooling_hours
```

This measures **degrees of temperature gap maintained per hour of cooling runtime**. Higher = more efficient (bigger gap with less runtime).

**Why this metric:**
- Controls for weather by normalizing against the indoor-outdoor temperature difference
- Uses only cooling runtime (heating is irrelevant in this summer-dominant dataset)
- Daily grain smooths out 15-minute cycling noise

**Why we filter `outdoor_temp > indoor_temp`:** When it’s cooler outside than inside, the AC running briefly isn’t a real cooling scenario — including those rows produces misleading negative efficiency values.

**Caveat:** This metric inflates on mild days when cooling hours are very low. A day with a 5°C gap and 0.8 hours of cooling scores higher than a day with a 12°C gap and 6 hours of cooling — but the latter represents a much harder working day. We’ll address this in the change point analysis.

In [0]:
spark.sql(f"""
SELECT device_id,
       DATE(CAST(interval_start AS TIMESTAMP)) AS day,
       AVG(CAST(outdoor_temp AS DOUBLE) - indoor_temp) AS avg_temp_gap_c,
       SUM(cooling_time) / 3600.0 AS cooling_hours,
       AVG(CAST(outdoor_temp AS DOUBLE) - indoor_temp) / NULLIF(SUM(cooling_time) / 3600.0, 0) AS cooling_efficiency
FROM {full_table_name}
WHERE cooling_time > 0
  AND outdoor_temp IS NOT NULL
  AND CAST(outdoor_temp AS DOUBLE) > indoor_temp
GROUP BY device_id, DATE(CAST(interval_start AS TIMESTAMP))
ORDER BY device_id, day
""").display()

Databricks visualization. Run in Databricks to view.

## 4. Detecting the Window Replacement

A window replacement improves insulation, which should show up as a **structural break** in the efficiency metric — a sustained shift in how much runtime is needed to maintain a given temperature gap.

**Approach:** For each candidate day, we compute the mean **cooling load ratio** (`cooling_hours / temp_gap`, the inverse of efficiency) before and after that day. The day with the largest combined drop across both devices is our predicted change point.

**Why load ratio instead of efficiency:** Efficiency (gap/hours) inflates when cooling hours are low (mild weather), producing false signals in October. Load ratio (hours/gap) is more stable — a drop means the system needs less runtime per degree of gap.

**Why both devices:** A whole-home window replacement should affect both thermostats simultaneously. A shift in only one device might indicate a thermostat setting change or a localized issue.

**Answer:** The window replacement was on **October 5, 2026** (confirmed by the homeowner).

In [0]:
import pandas as pd
from datetime import date

# Load daily metrics for change point detection
daily_cp = spark.sql(f"""
SELECT device_id,
       DATE(CAST(interval_start AS TIMESTAMP)) AS day,
       AVG(CAST(outdoor_temp AS DOUBLE) - indoor_temp) AS avg_temp_gap_c,
       SUM(cooling_time) / 3600.0 AS cooling_hours
FROM {full_table_name}
WHERE cooling_time > 0
  AND outdoor_temp IS NOT NULL
  AND CAST(outdoor_temp AS DOUBLE) > indoor_temp
GROUP BY device_id, DATE(CAST(interval_start AS TIMESTAMP))
ORDER BY device_id, day
""").toPandas()

daily_cp['load_ratio'] = daily_cp['cooling_hours'].astype(float) / daily_cp['avg_temp_gap_c']

# Change point analysis: for each candidate day, compute mean load ratio before vs after
results = []
all_days = sorted(daily_cp['day'].unique())

for cp in all_days:
    before = daily_cp[daily_cp['day'] < cp]
    after = daily_cp[daily_cp['day'] >= cp]
    if len(before) < 7 or len(after) < 7:
        continue
    shifts = []
    for dev in daily_cp['device_id'].unique():
        b = before[before['device_id'] == dev]['load_ratio'].mean()
        a = after[after['device_id'] == dev]['load_ratio'].mean()
        if pd.notna(b) and pd.notna(a):
            shifts.append(b - a)  # positive = ratio dropped (improvement)
    if len(shifts) == 2:
        results.append({
            'change_point': cp,
            'combined_drop': sum(shifts) / len(shifts),
        })

cp_df = pd.DataFrame(results).sort_values('combined_drop', ascending=False)

print("=== Top 5 Change Point Candidates (by combined drop in load ratio) ===")
print(f"{'Date':<14} {'Combined Drop':<16}")
print("-" * 32)
for _, row in cp_df.head(5).iterrows():
    marker = " ◄ actual date" if str(row['change_point']) == "2026-10-05" else ""
    print(f"{str(row['change_point']):<14} {row['combined_drop']:+.4f}{marker}")

## 5. Before vs After: What Changed

With the window replacement date established (Oct 5), we compare all metrics before and after. The challenge: October weather is milder than August, so raw before/after comparisons conflate weather with window effects.

**Two approaches:**
1. **Aggregate comparison** (below) — overall averages before vs after (confounded by weather)
2. **Like-for-like matching** (next cell) — find days with similar outdoor temperatures before and after Oct 5, then compare. This controls for weather, isolating the window effect.

The final cell checks whether thermostat settings changed — critical for interpreting whether the system is working differently because of the windows or because the homeowner changed the setpoint.

In [0]:
from pyspark.sql.functions import col, when, avg, sum as spark_sum, count, date_trunc

# Window replacement date
window_date = "2026-10-05"

# Comprehensive before/after analysis across all metrics
spark.sql(f"""
WITH daily AS (
  SELECT device_id,
         DATE(CAST(interval_start AS TIMESTAMP)) AS day,
         AVG(indoor_temp) AS avg_indoor_temp,
         AVG(CAST(outdoor_temp AS DOUBLE)) AS avg_outdoor_temp,
         AVG(indoor_humidity) AS avg_indoor_humidity,
         AVG(CAST(outdoor_humidity AS DOUBLE)) AS avg_outdoor_humidity,
         SUM(cooling_time) / 3600.0 AS cooling_hours,
         SUM(heating_time) / 3600.0 AS heating_hours,
         SUM(fan_time) / 3600.0 AS fan_hours,
         SUM(off_mode_time) / 3600.0 AS off_hours,
         COUNT(*) AS intervals
  FROM {full_table_name}
  GROUP BY device_id, DATE(CAST(interval_start AS TIMESTAMP))
)
SELECT device_id,
       CASE WHEN day < '{window_date}' THEN 'Before (Aug 1 – Oct 4)'
            ELSE 'After (Oct 5 – Oct 13)' END AS period,
       COUNT(*) AS days,
       ROUND(AVG(avg_indoor_temp), 2) AS avg_indoor_temp_c,
       ROUND(AVG(avg_outdoor_temp), 2) AS avg_outdoor_temp_c,
       ROUND(AVG(avg_indoor_humidity), 1) AS avg_indoor_humidity_pct,
       ROUND(AVG(avg_outdoor_humidity), 1) AS avg_outdoor_humidity_pct,
       ROUND(AVG(cooling_hours), 2) AS avg_cooling_hours,
       ROUND(AVG(fan_hours), 2) AS avg_fan_hours,
       ROUND(AVG(off_hours), 2) AS avg_off_hours,
       ROUND(AVG(cooling_hours) / NULLIF(AVG(avg_outdoor_temp - avg_indoor_temp), 0), 4) AS avg_load_ratio
FROM daily
WHERE cooling_hours > 0
  AND avg_outdoor_temp > avg_indoor_temp
GROUP BY device_id,
         CASE WHEN day < '{window_date}' THEN 'Before (Aug 1 – Oct 4)'
              ELSE 'After (Oct 5 – Oct 13)' END
ORDER BY device_id, period
""").display()

In [0]:
from pyspark.sql.functions import col, when, avg, sum as spark_sum, count, date_trunc
import pandas as pd

window_date = "2026-10-05"

# Get daily metrics for all days (including non-cooling days) to compare like-for-like
daily_pdf = spark.sql(f"""
WITH daily AS (
  SELECT device_id,
         DATE(CAST(interval_start AS TIMESTAMP)) AS day,
         AVG(indoor_temp) AS avg_indoor_temp,
         AVG(CAST(outdoor_temp AS DOUBLE)) AS avg_outdoor_temp,
         AVG(indoor_humidity) AS avg_indoor_humidity,
         AVG(CAST(outdoor_humidity AS DOUBLE)) AS avg_outdoor_humidity,
         SUM(cooling_time) / 3600.0 AS cooling_hours,
         SUM(fan_time) / 3600.0 AS fan_hours,
         SUM(off_mode_time) / 3600.0 AS off_hours
  FROM {full_table_name}
  GROUP BY device_id, DATE(CAST(interval_start AS TIMESTAMP))
)
SELECT *,
       avg_outdoor_temp - avg_indoor_temp AS temp_gap
FROM daily
ORDER BY device_id, day
""").toPandas()

daily_pdf['after_window'] = daily_pdf['day'] >= pd.Timestamp(window_date).date()

# For each device, find days with similar outdoor temps (within 2°C) before and after
print("=== Like-for-like comparison: days with similar outdoor temperature ===\n")
for dev_name, dev_id in [('Upstairs', '09AA01AF171903L1'), ('Downstairs', '15AA01AF062008H1')]:
    dev_df = daily_pdf[daily_pdf['device_id'] == dev_id].copy()
    
    after = dev_df[dev_df['after_window']]
    before = dev_df[~dev_df['after_window']]
    
    print(f"--- {dev_name} ---")
    print(f"{'Date':<12} {'Period':<8} {'Out T':<8} {'In T':<8} {'Gap':<8} {'Cool Hrs':<10} {'Indoor H%':<10} {'Out H%':<10}")
    print("-" * 76)
    
    for _, a_row in after.iterrows():
        # Find closest before-day by outdoor temp
        before['temp_diff'] = abs(before['avg_outdoor_temp'] - a_row['avg_outdoor_temp'])
        best_match = before.loc[before['temp_diff'].idxmin()]
        
        for row, label in [(best_match, 'Before'), (a_row, 'After')]:
            gap = row['avg_outdoor_temp'] - row['avg_indoor_temp']
            print(f"{str(row['day']):<12} {label:<8} {row['avg_outdoor_temp']:.1f}    {row['avg_indoor_temp']:.1f}    {gap:.2f}   {row['cooling_hours']:.2f}      {row['avg_indoor_humidity']:.1f}      {row['avg_outdoor_humidity']:.1f}")
        
        cool_delta = a_row['cooling_hours'] - best_match['cooling_hours']
        gap_delta = (a_row['avg_outdoor_temp'] - a_row['avg_indoor_temp']) - (best_match['avg_outdoor_temp'] - best_match['avg_indoor_temp'])
        print(f"  → Cooling hours change: {cool_delta:+.2f} hrs/day  |  Temp gap change: {gap_delta:+.2f}°C  |  Indoor humidity change: {(a_row['avg_indoor_humidity'] - best_match['avg_indoor_humidity']):+.1f}%")
        print()

In [0]:
# Did the thermostat settings change? And is the system now hitting its target?
print("=== Thermostat Settings & Target Achievement (during cooling) ===\n")
spark.sql(f"""
SELECT device_id,
       CASE WHEN DATE(CAST(interval_start AS TIMESTAMP)) < '2026-10-05' THEN 'Before'
            ELSE 'After' END AS period,
       ROUND(AVG(cooling_target), 2) AS avg_cooling_target,
       ROUND(AVG(indoor_temp), 2) AS avg_indoor_temp,
       ROUND(AVG(indoor_temp) - AVG(cooling_target), 2) AS temp_above_target,
       ROUND(AVG(indoor_humidity), 1) AS avg_indoor_humidity,
       ROUND(AVG(CAST(outdoor_humidity AS DOUBLE) - indoor_humidity), 1) AS humidity_barrier
FROM {full_table_name}
WHERE cooling_time > 0
  AND CAST(outdoor_temp AS DOUBLE) > indoor_temp
GROUP BY device_id,
         CASE WHEN DATE(CAST(interval_start AS TIMESTAMP)) < '2026-10-05' THEN 'Before'
              ELSE 'After' END
ORDER BY device_id, period
""").display()

## 6. Key Findings

### The windows didn’t reduce energy use — they made the cooling **effective**

| Metric | Before (Aug 1 – Oct 4) | After (Oct 5 – Oct 13) | Change |
| --- | --- | --- | --- |
| Upstairs — temp above target | +0.78°C (overshooting) | +0.31°C (nailing it) | System now hits setpoint |
| Upstairs — indoor humidity | 61.0% | 66.3% | +5.3% (better sealing traps moisture) |
| Downstairs — humidity barrier | −2.3% (indoor > outdoor) | +0.1% (equal) | Windows block outdoor moisture |
| Cooling runtime per interval | 0.18 hrs | 0.17 hrs | Flat — same effort, better result |

### What actually changed

1. **The AC now hits its target.** Before Oct 5, the Upstairs thermostat was set to 24.5°C but the house sat at 25.3°C — the AC was running but losing the battle against heat infiltration. After Oct 5, the same target yields 24.9°C. The windows reduced heat gain enough that the AC can keep up.

2. **Humidity dynamics shifted.** Upstairs humidity rose 5 points (better sealing traps indoor moisture). Downstairs humidity stayed flat despite higher outdoor humidity — the new windows are blocking moisture infiltration.

3. **The efficiency metric was misleading.** The “gap per cooling hour” looked *worse* after Oct 5 because October is milder (smaller temp gaps inflate the ratio’s denominator). The real improvement is in **comfort** — the system achieves its setpoint instead of overshooting it.

### Lessons for HVAC data analysis

* **Always control for weather.** Raw before/after comparisons conflate seasonal cooling with structural changes. Match days by outdoor temperature for a fair comparison.
* **Check thermostat settings.** A change in cooling target looks identical to an efficiency improvement in the data. Always verify the setpoint didn’t change.
* **“Efficiency” depends on the metric.** Gap-per-hour rewards mild weather; hours-per-gap penalizes it. Neither captures comfort — whether the system actually achieves its target.
* **Change point detection doesn’t need ML.** A simple before/after mean comparison across all candidate days is enough to surface structural breaks when you have 60+ days of data.